In [1]:
# wandbのライブラリをimport
import wandb

# wandbへログイン
wandb.login()

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/hfukumor/.netrc
wandb: Currently logged in as: hideo-fukumori (hideo-fukumori-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
# Pandasのライブラリをインポート
import pandas as pd

# 学習データを読み込んで変数 train に格納
train = pd.read_csv('./train.csv')

# 学習データの表示
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# テストデータを読み込んで変数 test に格納
test = pd.read_csv('./test.csv')

# テストデータの表示
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
# 学習データとテストデータ数を確認
print(train.shape)
print(test.shape)

(891, 12)
(418, 11)


In [5]:
# trainの欠損値の数を調査して、表示する
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [6]:
# testの欠損値の数を調査して、表示する
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [7]:
# 新しい空のDataFrameを作成する
temp = pd.DataFrame()

# 学習データとテストデータのAge列を連結させ、tempにAge列として追加する
temp['Age'] = pd.concat([train['Age'], test['Age']])

# trainとtestのAge列について、欠損値をtempのAge列の平均値で埋める
train['Age'] = train['Age'].fillna(temp['Age'].mean())
test['Age'] = test['Age'].fillna(temp['Age'].mean())

In [8]:
# 学習データとテストデータのFare列を連結させ、tempにFare列として追加する
temp['Fare'] = pd.concat([train['Fare'], test['Fare']])

# testのみに存在するFare列の欠損値をtempのFare列の平均値で埋める
test['Fare'] = test['Fare'].fillna(temp['Fare'].mean())

In [9]:
# 学習データとテストデータのEmbarked列を連結させ、tempにEmbarked列として追加する
temp['Embarked'] = pd.concat([train['Embarked'], test['Embarked']])

# tempのEmbakedの値を集計する
temp['Embarked'].value_counts()

Embarked
S    914
C    270
Q    123
Name: count, dtype: int64

In [10]:
# trainのみに存在するEmbarked列の欠損値を'S'で埋める
train['Embarked'] = train['Embarked'].fillna('S')

In [11]:
# trainとtestから、Cabin,Name,Ticket列を削除する
train = train.drop(columns=['Cabin', 'Name', 'Ticket'])
test = test.drop(columns=['Cabin', 'Name', 'Ticket'])

In [12]:
# trainのSex列とEmbarked列をダミー変数化して、変数train2に格納する
train2 = pd.get_dummies(data=train, columns=['Sex', 'Embarked'])

# testのSex列とEmbarked列をダミー変数化して、変数test2に格納する
test2 = pd.get_dummies(data=test, columns=['Sex', 'Embarked'])

In [13]:
# train2の欠損値の数を調査して、表示する
train2.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Age            0
SibSp          0
Parch          0
Fare           0
Sex_female     0
Sex_male       0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
dtype: int64

In [14]:
# test2の欠損値の数を調査して、表示する
test2.isnull().sum()

PassengerId    0
Pclass         0
Age            0
SibSp          0
Parch          0
Fare           0
Sex_female     0
Sex_male       0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
dtype: int64

In [15]:
# train2の各列の型を表示する
train2.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Sex_female        bool
Sex_male          bool
Embarked_C        bool
Embarked_Q        bool
Embarked_S        bool
dtype: object

In [16]:
# numpyのimport
import numpy as np

# train2をX_trainとY_trainに分ける
X_train = np.array(train2.drop(columns=['Survived'])).astype('float32')
Y_train = np.array(train2['Survived']).astype('float32')

# test2のデータ全体をX_testに格納する
X_test = np.array(test2).astype('float32')

In [17]:
# X_trainとY_trainの3割をX_validとY_validに分割する
from sklearn.model_selection import train_test_split

X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.3, random_state=0)

In [18]:
# 学習データと検証データ、テストデータの形状を確認
print("X_train=", X_train.shape, ", Y_train=", Y_train.shape)
print("X_valid=", X_valid.shape, ", Y_valid=", Y_valid.shape)
print("X_test=", X_test.shape)

X_train= (623, 11) , Y_train= (623,)
X_valid= (268, 11) , Y_valid= (268,)
X_test= (418, 11)


In [18]:
# tensorflowのimport
import tensorflow as tf

In [19]:
# モデルの構築と学習を定義する関数
def train_model():
    # wandbの初期設定
    wandb.init(
        # wandbでのプロジェクト名
        project="kaggle-titanic",
        # wandbで記録してもらいたい設定値
        config={
            "input_dense_shape": 8,
            "hidden_dense_shape": 8,
            "optimizer": "rmsprop",
            "batch_size": 32
        })

    # モデルの初期化とレイヤー定義
    model = tf.keras.Sequential([
        # 入力層 (Inputオブジェクトを使用)
        tf.keras.Input(shape=(11,)),
        tf.keras.layers.Dense(wandb.config.input_dense_shape, activation='relu'),
        # 隠れ層
        tf.keras.layers.Dense(wandb.config.hidden_dense_shape, activation='relu'),
        # 出力層
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    # モデルの構築
    model.compile(optimizer=wandb.config.optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # 学習の実施
    log = model.fit(X_train, Y_train, epochs=5000, batch_size=wandb.config.batch_size, verbose=True,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                                min_delta=0, patience=100,
                                                                verbose=1),
                              wandb.keras.WandbMetricsLogger(log_freq='epoch')
                              ],
                    validation_data=(X_valid, Y_valid))

In [20]:
# train_modelを実行する
train_model()

Epoch 1/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4960 - loss: 2.4860 - val_accuracy: 0.6119 - val_loss: 1.0886
Epoch 2/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5907 - loss: 1.1773 - val_accuracy: 0.3694 - val_loss: 1.2425
Epoch 3/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5538 - loss: 1.0205 - val_accuracy: 0.6493 - val_loss: 0.9088
Epoch 4/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6116 - loss: 0.8543 - val_accuracy: 0.6418 - val_loss: 0.8123
Epoch 5/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6453 - loss: 0.7665 - val_accuracy: 0.6679 - val_loss: 0.7013
Epoch 6/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6260 - loss: 0.7235 - val_accuracy: 0.6903 - val_loss: 0.6578
Epoch 7/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6597 - loss: 0.6628 - val_accuracy: 0.6903 - val_loss: 0.6397
Epoch 8/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6421 - loss: 0.6529 - val_accuracy: 0.

In [21]:
# wandbの動作を終了させる
wandb.finish()

epoch/accuracy,▁▁▁▅▅▆▆▆▆▆▆▇▆▆▆▇▇▇▇▇▇█▇▇█▇▇▇▇▇▆▇▇▇▆██▇▇▇
epoch/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇████
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▄▄▄▃▄▃▃▂▃▃▂▂▂▂▂▂▂▃▂▂▂▂▃▂▂▂▂▂▁▂▂▂▁▁▂▁▂
epoch/val_accuracy,▃▅▃▆▆▄█▇▆▇▆▇██▇▇▆█▇█▇▇▇███▇██▇█▆█▇█▇▇██▁
epoch/val_loss,▆▃█▂▃▂▄▁▂▃▃▂▁▂▂▃▂▁▁▁▂▂▂▂▁▂▁▁▂▄▁▁▂▁▂▁▂▁▃▅
epoch/accuracy,0.80096
epoch/epoch,331
epoch/learning_rate,0.001
epoch/loss,0.47097
epoch/val_accuracy,0.78358


In [22]:
# wandbでsweep（グリッドサーチ）を行なうための設定
sweep_config = {
    'method': 'grid',
    'name': 'kaggle-titanic-sweep',
    'metric': {
        'goal': 'maximize',
        'name': 'accuracy'
    },
    'parameters': {
        'input_dense_shape': {'values': [8, 16, 24]},
        'hidden_dense_shape': {'values': [8, 16, 24]},
        'optimizer': {'values': ['sgd', 'rmsprop', 'adam']},
        'batch_size': {'values': [16, 32, 64]}
     }
}

# sweep_configの設定値でsweepを初期化する
sweep_id = wandb.sweep(sweep=sweep_config, project="kaggle-titanic")

Create sweep with ID: qrl8sn1z
Sweep URL: https://wandb.ai/hideo-fukumori-personal/kaggle-titanic/sweeps/qrl8sn1z


In [23]:
# sweepを開始する
wandb.agent(sweep_id, function=train_model)

wandb: Agent Starting Run: 8kbq0yf4 with config:
wandb: 	batch_size: 16
wandb: 	hidden_dense_shape: 8
wandb: 	input_dense_shape: 8
wandb: 	optimizer: sgd


Epoch 1/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6051 - loss: 2.5478 - val_accuracy: 0.6269 - val_loss: 0.6898
Epoch 2/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6116 - loss: 0.6869 - val_accuracy: 0.6269 - val_loss: 0.6855
Epoch 3/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 969us/step - accuracy: 0.6116 - loss: 0.6835 - val_accuracy: 0.6269 - val_loss: 0.6819
Epoch 4/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 922us/step - accuracy: 0.6116 - loss: 0.6808 - val_accuracy: 0.6269 - val_loss: 0.6790
Epoch 5/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 972us/step - accuracy: 0.6116 - loss: 0.6785 - val_accuracy: 0.6269 - val_loss: 0.6765
Epoch 6/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 927us/step - accuracy: 0.6116 - loss: 0.6767 - val_accuracy: 0.6269 - val_loss: 0.6744
Epoch 7/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 980us/step - accuracy: 0.6116 - loss: 0.6751 - val_accuracy: 0.6269 - val_loss: 0.6726
Epoch 8/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 919us/step - accuracy: 0.6116 - loss: 0.6739 - val_

epoch/accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/epoch,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.61156
epoch/epoch,321
epoch/learning_rate,0.01
epoch/loss,0.66737
epoch/val_accuracy,0.62687


wandb: Agent Starting Run: x6hhmys5 with config:
wandb: 	batch_size: 16
wandb: 	hidden_dense_shape: 8
wandb: 	input_dense_shape: 8
wandb: 	optimizer: rmsprop


Epoch 1/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5907 - loss: 2.5139 - val_accuracy: 0.6828 - val_loss: 1.4203
Epoch 2/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6533 - loss: 1.1384 - val_accuracy: 0.6791 - val_loss: 0.9473
Epoch 3/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6067 - loss: 0.9354 - val_accuracy: 0.6381 - val_loss: 0.8572
Epoch 4/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 956us/step - accuracy: 0.5955 - loss: 0.9151 - val_accuracy: 0.5336 - val_loss: 1.0308
Epoch 5/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 959us/step - accuracy: 0.6212 - loss: 0.7887 - val_accuracy: 0.7090 - val_loss: 0.7183
Epoch 6/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 947us/step - accuracy: 0.6035 - loss: 0.8473 - val_accuracy: 0.6306 - val_loss: 0.9201
Epoch 7/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 990us/step - accuracy: 0.5939 - loss: 0.7978 - val_accuracy: 0.4478 - val_loss: 1.2525
Epoch 8/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 953us/step - accuracy: 0.6196 - loss: 0.7954 - val_ac

...

epoch/accuracy,▁▂▂▄▄▅▅▅▅▅▅▆▅▅▆▆▅▆▅▇▆▆▇▆▆▇▆▆▆▇▆▇▇██▇█▇▇█
epoch/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▇▇▇▇▇▇████
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▂▅▃▁▃▁█▆▇▇▇▅▇▅▆▇▂▅▃▆█▇▆▂█▆▂█▇▆▇▆▇███▇█▅▆
epoch/val_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.7817
epoch/epoch,280
epoch/learning_rate,0.001
epoch/loss,0.53688
epoch/val_accuracy,0.59701


wandb: Agent Starting Run: sf8ircq1 with config:
wandb: 	batch_size: 64
wandb: 	hidden_dense_shape: 24
wandb: 	input_dense_shape: 24
wandb: 	optimizer: adam


Epoch 1/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4623 - loss: 5.5291 - val_accuracy: 0.6157 - val_loss: 2.6906
Epoch 2/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5474 - loss: 2.7352 - val_accuracy: 0.4328 - val_loss: 2.1181
Epoch 3/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5377 - loss: 1.6269 - val_accuracy: 0.4925 - val_loss: 1.3877
Epoch 4/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5586 - loss: 1.0962 - val_accuracy: 0.5933 - val_loss: 0.7679
Epoch 5/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6276 - loss: 0.8941 - val_accuracy: 0.6828 - val_loss: 0.6985
Epoch 6/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6260 - loss: 0.8218 - val_accuracy: 0.6940 - val_loss: 0.6692
Epoch 7/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6356 - loss: 0.7770 - val_accuracy: 0.6754 - val_loss: 0.6942
Epoch 8/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6421 - loss: 0.7530 - val_accuracy: 0.

epoch/accuracy,▁▃▄▅▆▅▆▅▄▄▆▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇▆█▇█████▆▇▇██
epoch/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▃▄▃▃▃▃▃▃▂▃▃▂▂▂▁▂▂▁▂▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁▂▁▂
epoch/val_accuracy,▂▁▇▅▅▅▆▇▇▇▃▆▇█▇▇▇▇▇▇▇▇███▇▇▇███▇▅██▇▇▇▇█
epoch/val_loss,▄▄▄█▃▃▃▂▂▂▂▂▁▁▂▂▃▁▁▂▂▂▃▁▂▁▁▁▁▂▂▂▂▁▁▃▂▄▃▂
epoch/accuracy,0.80738
epoch/epoch,192
epoch/learning_rate,0.001
epoch/loss,0.43646
epoch/val_accuracy,0.77612


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


In [24]:
# グリッドサーチで得た最適なパラメータ値でモデルを作り直す
# （wandbは利用しない）
# テストデータによる予測も行なう
def train_model():
    # モデルの初期化とレイヤー定義
    model = tf.keras.Sequential([
        # 入力層
        tf.keras.Input(shape=(11,)),
        tf.keras.layers.Dense(24, activation='relu'),
        # 隠れ層
        tf.keras.layers.Dense(8, activation='relu'),
        # 出力層
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    # モデルの構築
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # 学習の実施
    log = model.fit(X_train, Y_train,
                    epochs=5000,
                    batch_size=16,
                    verbose=True,
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(
                            monitor='val_loss',
                            min_delta=0,
                            patience=100,
                            verbose=1
                        )
                    ],
                    validation_data=(X_valid, Y_valid))

    # テストデータによる予測
    Y_pred_proba = model.predict(X_test)
    Y_pred = (Y_pred_proba > 0.5).astype("int32")

    # Y_predを返す
    return Y_pred

In [25]:
# train_modelを実行する
Y_pred = train_model()
Y_pred

Epoch 1/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3933 - loss: 25.2467 - val_accuracy: 0.3657 - val_loss: 15.9272
Epoch 2/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3868 - loss: 10.4078 - val_accuracy: 0.3657 - val_loss: 4.7336
Epoch 3/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 951us/step - accuracy: 0.4559 - loss: 1.5619 - val_accuracy: 0.4925 - val_loss: 0.8626
Epoch 4/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 901us/step - accuracy: 0.5811 - loss: 0.7718 - val_accuracy: 0.5784 - val_loss: 0.7586
Epoch 5/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 881us/step - accuracy: 0.6019 - loss: 0.7100 - val_accuracy: 0.6007 - val_loss: 0.7202
Epoch 6/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6132 - loss: 0.6869 - val_accuracy: 0.6007 - val_loss: 0.7126
Epoch 7/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 857us/step - accuracy: 0.6164 - loss: 0.6824 - val_accuracy: 0.6045 - val_loss: 0.7178
Epoch 8/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 872us/step - accuracy: 0.6180 - loss: 0.6789 - val

array([[0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
    

In [26]:
# X_testをDataFrameに戻し、X_test2に格納
X_test2 = pd.DataFrame(X_test, columns=test2.columns)

In [27]:
# Kaggleへ提出するためのデータが入ったDataFrameを作成
submission_data = pd.DataFrame()
submission_data["PassengerId"] = X_test2["PassengerId"].astype("int32")
submission_data["Survived"] = Y_pred
submission_data

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [28]:
# Kaggleに提出するためのCSVファイルを作成
submission_data.to_csv("my_submission_titanic.csv", index=False)